# 👁️ RetinalVision: InceptionV3 Model Training & Export Pipeline
Bu notebook; 4.127 retinal fundus görüntüsü üzerinde şampiyon mimari olan **InceptionV3** modelini eğitir, test eder ve Hugging Face Spaces / Gradio çıkarımı için  formatında kaydeder.

In [ ]:
# 1. Google Drive Bağlantısı
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Kütüphanelerin Yüklenmesi
import os
import numpy as np
import cv2
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from google.colab import files

print("TensorFlow Sürümü:", tf.__version__)
print("GPU Durumu:", tf.config.list_physical_devices('GPU'))

In [ ]:
# 3. Veri Setinin Yüklenmesi (225x225 Piksel)
# Google Drive'daki klasör yolunu kontrol edin:
DATADIR = "/content/drive/MyDrive/RetinalImaging/Eyes/"
if not os.path.exists(DATADIR):
    DATADIR = "/content/drive/MyDrive/DerinOgrenme/RetinalImaging/Eyes/"

CATEGORIES = ["Cataract", "Diabetic Retinopathy", "Glaucoma", "Normal"]
IMG_SIZE = 225

all_data = []
print("Görüntüler yükleniyor ve yeniden boyutlandırılıyor...")
for category in CATEGORIES:
    path = os.path.join(DATADIR, category)
    class_num = CATEGORIES.index(category)
    count = 0
    for img in os.listdir(path):
        try:
            img_array = cv2.imread(os.path.join(path, img))
            if img_array is not None:
                new_array = cv2.resize(img_array, (IMG_SIZE, IMG_SIZE))
                all_data.append([new_array, class_num])
                count += 1
        except Exception as e:
            pass
    print(f"  {category}: {count} görsel yüklendi.")

X, y = [], []
for features, label in all_data:
    X.append(features)
    y.append(label)

X = np.array(X, dtype="float32") / 255.0
y = to_categorical(np.array(y), num_classes=len(CATEGORIES))
print("Toplam Veri Sayısı:", X.shape[0])

In [ ]:
# 4. Eğitim ve Test Ayrımı (%80 Eğitim, %20 Test)
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print("Eğitim Seti:", x_train.shape)
print("Test Seti:", x_test.shape)

In [ ]:
# 5. InceptionV3 Model Mimarisi
base_model = InceptionV3(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(1024, activation="relu"),
    Dropout(0.5),
    Dense(len(CATEGORIES), activation="softmax")
])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
# 6. Modelin Eğitimi
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, min_lr=0.00001, verbose=1)
early_stop = EarlyStopping(monitor="val_loss", min_delta=0, patience=5, verbose=1, restore_best_weights=True)

history = model.fit(
    x_train, y_train,
    epochs=30,
    batch_size=32,
    validation_data=(x_test, y_test),
    callbacks=[reduce_lr, early_stop]
)

In [ ]:
# 7. Model Performansının Değerlendirilmesi
y_pred = model.predict(x_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

acc = accuracy_score(y_true, y_pred_classes)
print(f"🏆 Test Doğruluğu (Accuracy): %{acc*100:.2f}")
print("
Sınıflandırma Raporu:")
print(classification_report(y_true, y_pred_classes, target_names=CATEGORIES))

In [ ]:
# 8. Modelin Kaydedilmesi ve İndirilmesi
output_name = "retinal_inceptionv3.keras"
model.save(output_name)
print(f"✅ Model başarıyla {output_name} olarak kaydedildi.")

# Doğrudan bilgisayarınıza indirmek için:
files.download(output_name)